# 08 — Develop resistance phenotype rules

The reporting groups and drug vocabulary are defined. The next question is which resistance phenotypes can be derived from ARMD's categorical susceptibility results.

I treated the source designation (for example, `MRSA`, `VANCO RESISTANT`, or `CARBAPENEM RESISTANT`) as provenance, not as the final classification. The analytical rule uses reconciled AST and explicitly distinguishes **positive**, **not detected**, and **not evaluable** evidence. The source does not provide MICs, carbapenemase results, or a confirmed ESBL test for this analysis.

### CRE — identify the candidate source labels

## Reporting scope and saved evidence

This notebook documents upstream source exploration or model development across the **full ARMD release**. The optional 2023 reporting window is applied downstream; it does not turn these source-level counts into annual antibiogram denominators. Saved counts describe the source/model state when each cell was run. Mapping and reference-seed changes may require a fresh comparison.

The current susceptibility design independently selects the first qualifying culture within each reporting category, with a separate overall *S. aureus* branch. See [Notebook 13](13_validate_phenotype_stratified_antibiogram.ipynb) for the current design and saved 2023 coverage, [Notebook 14](14_explore_reporting_windows.ipynb) for window availability, and [metric definitions](../../docs/metric_definitions.md) for denominators and scope. SQL cells, execution counts, and existing outputs are preserved in this documentation revision.

In [1]:
%sql
with source_organisms as (

    select
        cohort_id,
        organism_source,

        count(*) as culture_organism_rows,
        count(distinct patient_key) as patients

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    group by
        cohort_id,
        organism_source

),

classified as (

    select
        cohort_id,
        organism_source,
        culture_organism_rows,
        patients,

        case

            -- Current and phenotype-qualified E. coli labels.
            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            -- Historical and current Klebsiella aerogenes labels.
            -- This condition must come before the broad
            -- Enterobacter and Klebsiella conditions.
            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            -- Other Klebsiella source labels.
            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            -- Enterobacter source labels other than historical
            -- Enterobacter aerogenes.
            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            -- Citrobacter source labels, including historical
            -- labels beginning with ZZZ.
            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            -- Only Serratia marcescens is included under the
            -- accepted CRE phenotype scope.
            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            -- Proteus species, including historical labels.
            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            -- Morganella scope is limited to M. morganii.
            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_organism_group

    from source_organisms

)

select
    cohort_id,
    cre_organism_group,
    organism_source,
    culture_organism_rows,
    patients

from classified

where cre_organism_group is not null

order by
    cohort_id,
    cre_organism_group,
    culture_organism_rows desc,
    organism_source;

,cohort_id,cre_organism_group,organism_source,culture_organism_rows,patients
0,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX,59,56
1,blood,Citrobacter spp.,CITROBACTER KOSERI,40,39
2,blood,Citrobacter spp.,CITROBACTER FREUNDII,19,18
3,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),9,9
4,blood,Citrobacter spp.,CITROBACTER KOSERI (CARBAPENEM RESISTANT),6,6
5,blood,Citrobacter spp.,CITROBACTER SPECIES,2,2
6,blood,Citrobacter spp.,CITROBACTER YOUNGAE,2,2
7,blood,Citrobacter spp.,ZZZCITROBACTER AMALONATICUS,2,2
8,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,254,240
9,blood,Enterobacter spp.,ENTEROBACTER CLOACAE,63,62


This first query is a discovery check on source labels, not the eligibility logic in the production model.

### CRE — check the reviewed mapping coverage

In [1]:
%sql
with source_labels as (

    select distinct
        organism_source,
        mapping_status

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

classified as (

    select
        organism_source,
        mapping_status,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group

    from source_labels

),

unmapped as (

    select
        organism_source,
        cre_rule_group,

        case
            when organism_source like '%(CARBAPENEM RESISTANT)%'
                then 1
            else 0
        end as source_designated_cre

    from classified

    where cre_rule_group is not null
      and mapping_status = 'unmapped'

)

select
    cre_rule_group,

    case
        when cre_rule_group in (
            'Proteus spp.',
            'Morganella morganii'
        )
            then 1
        else 0
    end as exclude_imipenem_from_cre_rule,

    count(*) as unmapped_source_label_count,

    array_sort(
        collect_set(organism_source)
    ) as unmapped_source_labels,

    array_sort(
        collect_set(
            case
                when source_designated_cre = 1
                    then organism_source
            end
        )
    ) as source_designated_cre_labels

from unmapped

group by cre_rule_group

order by cre_rule_group

,cre_rule_group,exclude_imipenem_from_cre_rule,unmapped_source_label_count,unmapped_source_labels,source_designated_cre_labels
0,Citrobacter spp.,0,11,"[CITROBACTER AMALONATICUS COMPLEX, CITROBACTER BRAAKII, CITROBACTER FREUNDII, CITROBACTER FREUNDII COMPLEX, CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT), CITROBACTER KOSERI, CITROBACTER KOSERI (CARBAPENEM RESISTANT), CITROBACTER SPECIES, CITROBACTER YOUNGAE, ZZZCITROBACTER AMALONATICUS, ZZZCITROBACTER FARMERI]","[CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT), CITROBACTER KOSERI (CARBAPENEM RESISTANT)]"
1,Enterobacter spp.,0,7,"[ENTEROBACTER AMNIGENUS 1, ENTEROBACTER ASBURIAE, ENTEROBACTER CANCEROGENUS, ENTEROBACTER CLOACAE, ENTEROBACTER GERGOVIAE, ENTEROBACTER SPECIES, ENTEROBACTER TAYLORAE]",[]
2,Klebsiella aerogenes,0,3,"[KLEBSIELLA AEROGENES, KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT), ZZZENTEROBACTER AEROGENES]",[KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT)]
3,Klebsiella spp.,0,3,"[KLEBSIELLA OZAENAE, KLEBSIELLA PNEUMONIAE SSP. OZAENAE, KLEBSIELLA SPECIES]",[]
4,Morganella morganii,1,1,[MORGANELLA MORGANII],[]
5,Proteus spp.,1,4,"[PROTEUS SPECIES, PROTEUS VULGARIS GROUP, ZZZPROTEUS PENNERI, ZZZPROTEUS VULGARIS]",[]


Patterns in source names help find candidates, but the production rule should use the reviewed mapping. This check flags labels that could be relevant to CRE but still need terminology review.

### CRE — inspect the actual reporting groups

In [ ]:
%sql
with vocabulary as (

    select
        organism_source,
        max(organism_standardized) as organism_standardized,
        max(organism_reporting_group) as organism_reporting_group,
        max(source_qualifier) as source_qualifier,
        max(mapping_status) as mapping_status,
        count(*) as culture_organism_rows,
        count(distinct patient_key) as patients,
        array_sort(collect_set(cohort_id)) as cohorts
    from workspace.micro_dev.int_mapped_cohort_culture_organism
    where organism_source like 'ENTEROBACTER %'
       or organism_source like 'ZZZENTEROBACTER %'
       or organism_source like 'KLEBSIELLA AEROGENES%'
       or organism_source like 'CITROBACTER %'
       or organism_source like 'ZZZCITROBACTER %'
    group by organism_source

)
select
    organism_source,
    organism_standardized,
    organism_reporting_group,
    source_qualifier,
    mapping_status,
    case
        when upper(organism_source) like '%CARBAPENEM RESISTANT%'
            then 1
        else 0
    end as source_designated_cre,
    culture_organism_rows,
    patients,
    cohorts
from vocabulary
order by
    organism_reporting_group,
    organism_source;

The final reporting taxonomy is taken from `organism_reporting_group`. In particular, *Citrobacter koseri* and *C. farmeri* remain separate from the *C. freundii* complex. The query uses the current mapping rather than an exploratory proposed grouping.

### CRE — compare source designation with AST evidence

In [10]:
%sql
with organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group,

        case
            when upper(organism_source) like '%CARBAPENEM RESISTANT%'
                then 1
            else 0
        end as source_designated_cre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

cre_eligible as (

    select *
    from organism_scope
    where cre_rule_group is not null

),

qualifying_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,
        e.organism_source,
        e.cre_rule_group,
        e.source_designated_cre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from cre_eligible e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key

       and (

            (
                e.cre_rule_group in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam'
                )
            )

            or

            (
                e.cre_rule_group not in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Imipenem',
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam',
                    'Imipenem/Relebactam'
                )
            )

       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as qualifying_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as qualifying_carbapenem_results

    from qualifying_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_cre_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_cre = 1
             and ast_cre_status = 'positive'
                then 'concordant_positive'

            when source_designated_cre = 1
             and ast_cre_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_cre = 1
             and ast_cre_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_cre = 0
             and ast_cre_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_cre = 0
             and ast_cre_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as cre_concordance_status

    from ast_classified

)

select
    cohort_id,
    cre_rule_group,
    cre_concordance_status,

    count(*) as culture_organism_records,

    sum(source_designated_cre)
        as source_designated_cre_records,

    sum(
        case
            when ast_cre_status = 'positive' then 1
            else 0
        end
    ) as ast_cre_positive_records

from concordance

group by
    cohort_id,
    cre_rule_group,
    cre_concordance_status

order by
    cohort_id,
    cre_rule_group,
    cre_concordance_status

,cohort_id,cre_rule_group,cre_concordance_status,culture_organism_records,source_designated_cre_records,ast_cre_positive_records
0,blood,Citrobacter spp.,no_source_label_ast_not_detected,123,0,0
1,blood,Citrobacter spp.,no_source_label_not_evaluable,1,0,0
2,blood,Citrobacter spp.,source_positive_ast_not_detected,15,15,0
3,blood,Enterobacter spp.,ast_positive_without_source_label,8,0,8
4,blood,Enterobacter spp.,concordant_positive,3,3,3
5,blood,Enterobacter spp.,no_source_label_ast_not_detected,320,0,0
6,blood,Escherichia coli,ast_positive_without_source_label,1,0,1
7,blood,Escherichia coli,concordant_positive,4,4,4
8,blood,Escherichia coli,no_source_label_ast_not_detected,3334,0,0
9,blood,Escherichia coli,no_source_label_not_evaluable,1,0,0


CRE-positive evidence comes from a resolved resistant result to a qualifying carbapenem in an eligible Enterobacterales group. Imipenem is excluded for the *Proteus* and *Morganella* rule groups. Without sufficient usable evidence, the record stays `not_evaluable` rather than becoming a negative.

### CRE — examine disagreements

In [11]:
%sql
with organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group,

        case
            when upper(organism_source) like '%CARBAPENEM RESISTANT%'
                then 1
            else 0
        end as source_designated_cre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

cre_eligible as (

    select *
    from organism_scope
    where cre_rule_group is not null

),

qualifying_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,
        e.organism_source,
        e.cre_rule_group,
        e.source_designated_cre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from cre_eligible e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key

       and (

            (
                e.cre_rule_group in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam'
                )
            )

            or

            (
                e.cre_rule_group not in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Imipenem',
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam',
                    'Imipenem/Relebactam'
                )
            )

       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as qualifying_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as qualifying_carbapenem_results

    from qualifying_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_cre_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_cre = 1
             and ast_cre_status = 'positive'
                then 'concordant_positive'

            when source_designated_cre = 1
             and ast_cre_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_cre = 1
             and ast_cre_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_cre = 0
             and ast_cre_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_cre = 0
             and ast_cre_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as cre_concordance_status

    from ast_classified

)

select
    cohort_id,
    cre_rule_group,
    organism_source,

    ast_cre_status,
    cre_concordance_status,

    qualifying_endpoint_count,
    evaluable_endpoint_count,
    non_resistant_endpoint_count,
    non_evaluable_endpoint_count,
    resistant_endpoint_count,

    concat_ws(
        ', ',
        qualifying_carbapenem_results
    ) as qualifying_carbapenem_result_pattern,

    count(*) as culture_organism_records

from concordance

where cre_concordance_status in (
    'source_positive_ast_not_detected',
    'source_positive_not_evaluable',
    'ast_positive_without_source_label'
)

group by
    cohort_id,
    cre_rule_group,
    organism_source,
    ast_cre_status,
    cre_concordance_status,
    qualifying_endpoint_count,
    evaluable_endpoint_count,
    non_resistant_endpoint_count,
    non_evaluable_endpoint_count,
    resistant_endpoint_count,
    qualifying_carbapenem_results

order by
    cohort_id,
    culture_organism_records desc,
    cre_rule_group,
    organism_source,
    qualifying_carbapenem_result_pattern

,cohort_id,cre_rule_group,organism_source,ast_cre_status,cre_concordance_status,qualifying_endpoint_count,evaluable_endpoint_count,non_resistant_endpoint_count,non_evaluable_endpoint_count,resistant_endpoint_count,qualifying_carbapenem_result_pattern,culture_organism_records
0,blood,Citrobacter spp.,CITROBACTER KOSERI (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Imipenem=Susceptible, Meropenem=Susceptible",6
1,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,positive,ast_positive_without_source_label,3,3,2,0,1,"Ertapenem=Resistant, Imipenem=Susceptible, Meropenem=Susceptible",6
2,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Imipenem=Susceptible, Meropenem=Susceptible",5
3,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,3,3,3,0,0,"Ertapenem=Susceptible, Imipenem=Susceptible, Meropenem=Susceptible",2
4,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Ertapenem=Susceptible, Meropenem=Susceptible",2
5,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,positive,ast_positive_without_source_label,2,2,1,0,1,"Ertapenem=Resistant, Meropenem=Susceptible",2
6,blood,Escherichia coli,ESCHERICHIA COLI,positive,ast_positive_without_source_label,3,3,2,0,1,"Ertapenem=Resistant, Imipenem=Susceptible, Meropenem=Susceptible",1
7,blood,Klebsiella aerogenes,KLEBSIELLA AEROGENES,positive,ast_positive_without_source_label,2,2,1,0,1,"Ertapenem=Resistant, Meropenem=Susceptible",1
8,blood,Klebsiella spp.,KLEBSIELLA PNEUMONIAE,positive,ast_positive_without_source_label,3,3,1,0,2,"Ertapenem=Resistant, Meropenem/Vaborbactam=Susceptible, Meropenem=Resistant",1
9,blood,Klebsiella spp.,KLEBSIELLA PNEUMONIAE,positive,ast_positive_without_source_label,2,2,0,0,2,"Ertapenem=Resistant, Meropenem=Resistant",1


I inspect source-designated resistance alongside the qualifying endpoints to understand discordant records. A source-designated CRE name does not override the AST-derived result.

### Oxacillin-resistant *S. aureus* — endpoint availability

In [1]:
%sql
with staphylococcus_aureus as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        case
            when upper(
                coalesce(source_qualifier, '')
            ) = 'MRSA'

              or upper(organism_source) like '%MRSA%'

              or upper(organism_source)
                    like '%OXACILLIN RESISTANT%'

                then 1

            else 0
        end as source_designated_oxacillin_resistant

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Staphylococcus aureus'

),

defining_results as (

    select
        s.culture_organism_key,
        s.culture_key,
        s.cohort_id,
        s.patient_key,

        s.organism_source,
        s.source_qualifier,
        s.source_designated_oxacillin_resistant,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from staphylococcus_aureus s

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on s.culture_organism_key = r.culture_organism_key

       and r.antibiotic in (
           'Oxacillin',
           'Cefoxitin'
       )

)

select
    cohort_id,
    organism_source,
    source_qualifier,
    source_designated_oxacillin_resistant,

    coalesce(
        antibiotic,
        '[no defining endpoint]'
    ) as defining_antibiotic,

    coalesce(
        resolution_status,
        'not_present'
    ) as resolution_status,

    coalesce(
        susceptibility,
        '[null]'
    ) as susceptibility,

    count(
        distinct culture_organism_key
    ) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from defining_results

group by
    cohort_id,
    organism_source,
    source_qualifier,
    source_designated_oxacillin_resistant,

    coalesce(
        antibiotic,
        '[no defining endpoint]'
    ),

    coalesce(
        resolution_status,
        'not_present'
    ),

    coalesce(
        susceptibility,
        '[null]'
    )

order by
    cohort_id,
    source_designated_oxacillin_resistant desc,
    organism_source,
    defining_antibiotic,
    resolution_status,
    susceptibility

,cohort_id,organism_source,source_qualifier,source_designated_oxacillin_resistant,defining_antibiotic,resolution_status,susceptibility,culture_organism_records,patients
0,blood,STAPH AUREUS {MRSA},MRSA,1,Oxacillin,conflict,[null],3,3
1,blood,STAPH AUREUS {MRSA},MRSA,1,Oxacillin,resolved,Resistant,418,366
2,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),small_colony_or_other_morphotype,0,Oxacillin,resolved,Resistant,3,3
3,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),small_colony_or_other_morphotype,0,Oxacillin,resolved,Susceptible,4,4
4,blood,STAPHYLOCOCCUS AUREUS,None,0,Oxacillin,conflict,[null],1,1
5,blood,STAPHYLOCOCCUS AUREUS,None,0,Oxacillin,missing,[null],1,1
6,blood,STAPHYLOCOCCUS AUREUS,None,0,Oxacillin,resolved,Resistant,200,187
7,blood,STAPHYLOCOCCUS AUREUS,None,0,Oxacillin,resolved,Susceptible,1297,1204
8,blood,STAPHYLOCOCCUS AUREUS,None,0,[no defining endpoint],not_present,[null],13,13
9,respiratory,STAPH AUREUS {MRSA},MRSA,1,Oxacillin,conflict,[null],17,13


Oxacillin and cefoxitin are the defining AST endpoints for this project. This query includes ordinary and MRSA-qualified source labels so missingness and label concordance can be reviewed.

### Oxacillin resistance — derive the status

In [2]:
%sql
with staphylococcus_aureus as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        case
            when upper(
                coalesce(source_qualifier, '')
            ) = 'MRSA'

              or upper(organism_source) like '%MRSA%'

              or upper(organism_source)
                    like '%OXACILLIN RESISTANT%'

                then 1

            else 0
        end as source_designated_oxacillin_resistant

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Staphylococcus aureus'

),

defining_results as (

    select
        s.culture_organism_key,
        s.culture_key,
        s.cohort_id,
        s.patient_key,

        s.organism_source,
        s.source_qualifier,
        s.source_designated_oxacillin_resistant,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from staphylococcus_aureus s

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on s.culture_organism_key = r.culture_organism_key

       and r.antibiotic in (
           'Oxacillin',
           'Cefoxitin'
       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,
        source_designated_oxacillin_resistant,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as defining_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as defining_ast_results

    from defining_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        source_qualifier,
        source_designated_oxacillin_resistant

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_oxacillin_resistance_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_oxacillin_resistant = 1
             and ast_oxacillin_resistance_status = 'positive'
                then 'concordant_positive'

            when source_designated_oxacillin_resistant = 1
             and ast_oxacillin_resistance_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_oxacillin_resistant = 1
             and ast_oxacillin_resistance_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_oxacillin_resistant = 0
             and ast_oxacillin_resistance_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_oxacillin_resistant = 0
             and ast_oxacillin_resistance_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as oxacillin_resistance_concordance_status

    from ast_classified

)

select
    cohort_id,
    organism_source,
    source_qualifier,

    source_designated_oxacillin_resistant,
    ast_oxacillin_resistance_status,
    oxacillin_resistance_concordance_status,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from concordance

group by
    cohort_id,
    organism_source,
    source_qualifier,
    source_designated_oxacillin_resistant,
    ast_oxacillin_resistance_status,
    oxacillin_resistance_concordance_status

order by
    cohort_id,
    source_designated_oxacillin_resistant desc,
    organism_source,
    oxacillin_resistance_concordance_status

,cohort_id,organism_source,source_qualifier,source_designated_oxacillin_resistant,ast_oxacillin_resistance_status,oxacillin_resistance_concordance_status,culture_organism_records,patients
0,blood,STAPH AUREUS {MRSA},MRSA,1,positive,concordant_positive,418,366
1,blood,STAPH AUREUS {MRSA},MRSA,1,not_evaluable,source_positive_not_evaluable,3,3
2,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),small_colony_or_other_morphotype,0,positive,ast_positive_without_source_label,3,3
3,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),small_colony_or_other_morphotype,0,not_detected,no_source_label_ast_not_detected,4,4
4,blood,STAPHYLOCOCCUS AUREUS,None,0,positive,ast_positive_without_source_label,200,187
5,blood,STAPHYLOCOCCUS AUREUS,None,0,not_detected,no_source_label_ast_not_detected,1297,1204
6,blood,STAPHYLOCOCCUS AUREUS,None,0,not_evaluable,no_source_label_not_evaluable,15,15
7,respiratory,STAPH AUREUS {MRSA},MRSA,1,positive,concordant_positive,890,379
8,respiratory,STAPH AUREUS {MRSA},MRSA,1,not_detected,source_positive_ast_not_detected,2,2
9,respiratory,STAPH AUREUS {MRSA},MRSA,1,not_evaluable,source_positive_not_evaluable,22,18


A resolved resistant result to either defining drug is positive. Clean non-resistant results support `not_detected`; unresolved defining endpoints remain `not_evaluable` when there is no positive evidence.

### VRE — vancomycin endpoint availability

In [3]:
%sql
with enterococcus_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        organism_standardized,
        source_qualifier,

        case
            when upper(
                coalesce(source_qualifier, '')
            ) = 'VANCOMYCIN_RESISTANT'

              or upper(organism_source)
                    like '%VANCO RESISTANT%'

              or upper(organism_source)
                    like '%VANCOMYCIN RESISTANT%'

                then 1

            else 0
        end as source_designated_vre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized in (
        'Enterococcus faecalis',
        'Enterococcus faecium'
    )

),

vancomycin_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,

        e.organism_source,
        e.organism_standardized,
        e.source_qualifier,
        e.source_designated_vre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from enterococcus_scope e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key
       and r.antibiotic = 'Vancomycin'

)

select
    cohort_id,
    organism_standardized,
    organism_source,
    source_qualifier,
    source_designated_vre,

    coalesce(
        antibiotic,
        '[no vancomycin endpoint]'
    ) as defining_antibiotic,

    coalesce(
        resolution_status,
        'not_present'
    ) as resolution_status,

    coalesce(
        susceptibility,
        '[null]'
    ) as susceptibility,

    count(
        distinct culture_organism_key
    ) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from vancomycin_results

group by
    cohort_id,
    organism_standardized,
    organism_source,
    source_qualifier,
    source_designated_vre,

    coalesce(
        antibiotic,
        '[no vancomycin endpoint]'
    ),

    coalesce(
        resolution_status,
        'not_present'
    ),

    coalesce(
        susceptibility,
        '[null]'
    )

order by
    cohort_id,
    organism_standardized,
    source_designated_vre desc,
    organism_source,
    resolution_status,
    susceptibility

,cohort_id,organism_standardized,organism_source,source_qualifier,source_designated_vre,defining_antibiotic,resolution_status,susceptibility,culture_organism_records,patients
0,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS - VANCO RESISTANT,vancomycin_resistant,1,Vancomycin,resolved,Resistant,2,2
1,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,Vancomycin,resolved,Resistant,4,4
2,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,Vancomycin,resolved,Susceptible,562,524
3,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM - VANCO RESISTANT,vancomycin_resistant,1,Vancomycin,resolved,Inconclusive,1,1
4,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM - VANCO RESISTANT,vancomycin_resistant,1,Vancomycin,resolved,Resistant,67,62
5,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,Vancomycin,conflict,[null],1,1
6,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,Vancomycin,resolved,Resistant,85,77
7,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,Vancomycin,resolved,Susceptible,138,133
8,respiratory,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,Vancomycin,resolved,Susceptible,3,3
9,respiratory,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,Vancomycin,resolved,Resistant,1,1


I limit this analysis to the reviewed *E. faecalis* and *E. faecium* identities and check vancomycin coverage before deriving the phenotype.

### VRE — derive the status

In [4]:
%sql
with enterococcus_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        organism_standardized,
        source_qualifier,

        case
            when upper(
                coalesce(source_qualifier, '')
            ) = 'VANCOMYCIN_RESISTANT'

              or upper(organism_source)
                    like '%VANCO RESISTANT%'

              or upper(organism_source)
                    like '%VANCOMYCIN RESISTANT%'

                then 1

            else 0
        end as source_designated_vre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized in (
        'Enterococcus faecalis',
        'Enterococcus faecium'
    )

),

vancomycin_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,

        e.organism_source,
        e.organism_standardized,
        e.source_qualifier,
        e.source_designated_vre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from enterococcus_scope e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key
       and r.antibiotic = 'Vancomycin'

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        organism_standardized,
        source_qualifier,
        source_designated_vre,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as defining_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when antibiotic is null
                    then 1

                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is null
                        then '[no vancomycin endpoint]'

                    else concat(
                        antibiotic,
                        '=',
                        coalesce(
                            susceptibility,
                            upper(resolution_status)
                        )
                    )
                end
            )
        ) as defining_ast_results

    from vancomycin_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        organism_standardized,
        source_qualifier,
        source_designated_vre

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_vancomycin_resistance_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_vre = 1
             and ast_vancomycin_resistance_status = 'positive'
                then 'concordant_positive'

            when source_designated_vre = 1
             and ast_vancomycin_resistance_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_vre = 1
             and ast_vancomycin_resistance_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_vre = 0
             and ast_vancomycin_resistance_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_vre = 0
             and ast_vancomycin_resistance_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as vre_concordance_status

    from ast_classified

)

select
    cohort_id,
    organism_standardized,
    organism_source,
    source_qualifier,

    source_designated_vre,
    ast_vancomycin_resistance_status,
    vre_concordance_status,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from concordance

group by
    cohort_id,
    organism_standardized,
    organism_source,
    source_qualifier,

    source_designated_vre,
    ast_vancomycin_resistance_status,
    vre_concordance_status

order by
    cohort_id,
    organism_standardized,
    source_designated_vre desc,
    organism_source,
    vre_concordance_status

,cohort_id,organism_standardized,organism_source,source_qualifier,source_designated_vre,ast_vancomycin_resistance_status,vre_concordance_status,culture_organism_records,patients
0,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS - VANCO RESISTANT,vancomycin_resistant,1,positive,concordant_positive,2,2
1,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,positive,ast_positive_without_source_label,4,4
2,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,not_detected,no_source_label_ast_not_detected,562,524
3,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM - VANCO RESISTANT,vancomycin_resistant,1,positive,concordant_positive,67,62
4,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM - VANCO RESISTANT,vancomycin_resistant,1,not_evaluable,source_positive_not_evaluable,1,1
5,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,positive,ast_positive_without_source_label,85,77
6,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,not_detected,no_source_label_ast_not_detected,138,133
7,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,not_evaluable,no_source_label_not_evaluable,1,1
8,respiratory,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,not_detected,no_source_label_ast_not_detected,3,3
9,respiratory,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,positive,ast_positive_without_source_label,1,1


Resolved vancomycin resistance is positive. Resolved susceptible/intermediate results support `not_detected`; absent or unresolved evidence does not.

### MDR *P. aeruginosa* — antimicrobial category coverage

In [6]:
%sql
with category_definitions as (

    select *
    from values

        (
            1,
            'extended_spectrum_cephalosporins',
            'Cefepime'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftazidime'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftazidime/Avibactam'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftolozane/Tazobactam'
        ),

        (
            2,
            'fluoroquinolones',
            'Ciprofloxacin'
        ),
        (
            2,
            'fluoroquinolones',
            'Levofloxacin'
        ),

        (
            3,
            'aminoglycosides',
            'Tobramycin'
        ),

        (
            4,
            'carbapenems',
            'Imipenem'
        ),
        (
            4,
            'carbapenems',
            'Meropenem'
        ),
        (
            4,
            'carbapenems',
            'Imipenem/Relebactam'
        ),

        (
            5,
            'antipseudomonal_penicillin_combination',
            'Piperacillin/Tazobactam'
        ),

        (
            6,
            'siderophore_cephalosporin',
            'Cefiderocol'
        )

    as definitions(
        category_order,
        mdr_category,
        antibiotic
    )

),

categories as (

    select distinct
        category_order,
        mdr_category

    from category_definitions

),

pseudomonas_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Pseudomonas aeruginosa'

),

category_results as (

    select
        p.culture_organism_key,
        p.culture_key,
        p.cohort_id,
        p.patient_key,

        p.organism_source,
        p.source_qualifier,

        d.category_order,
        d.mdr_category,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from pseudomonas_scope p

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on p.culture_organism_key = r.culture_organism_key

    inner join category_definitions d
        on r.antibiotic = d.antibiotic

),

isolate_category_grid as (

    select
        p.culture_organism_key,
        p.culture_key,
        p.cohort_id,
        p.patient_key,

        p.organism_source,
        p.source_qualifier,

        c.category_order,
        c.mdr_category

    from pseudomonas_scope p

    cross join categories c

),

category_summary as (

    select
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,

        g.category_order,
        g.mdr_category,

        count(
            r.antibiotic
        ) as endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as positive_endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Susceptible'
                    then 1
                else 0
            end
        ) as susceptible_endpoint_count,

        sum(
            case
                when r.antibiotic is not null
                 and r.resolution_status in (
                     'missing',
                     'conflict'
                 )
                    then 1

                when r.antibiotic is not null
                 and r.resolution_status = 'resolved'
                 and (
                     r.susceptibility is null
                     or r.susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count

    from isolate_category_grid g

    left join category_results r
        on g.culture_organism_key = r.culture_organism_key
       and g.mdr_category = r.mdr_category

    group by
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,

        g.category_order,
        g.mdr_category

),

category_classified as (

    select
        *,

        case
            when positive_endpoint_count > 0
                then 'positive'

            when endpoint_count = 0
                then 'not_tested'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when susceptible_endpoint_count > 0
                then 'observed_non_positive'

            else 'not_evaluable'
        end as observed_category_status

    from category_summary

)

select
    cohort_id,
    organism_source,
    source_qualifier,

    category_order,
    mdr_category,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    sum(
        case
            when observed_category_status <> 'not_tested'
                then 1
            else 0
        end
    ) as category_tested_records,

    sum(
        case
            when observed_category_status = 'positive'
                then 1
            else 0
        end
    ) as category_positive_records,

    sum(
        case
            when observed_category_status =
                 'observed_non_positive'
                then 1
            else 0
        end
    ) as category_observed_non_positive_records,

    sum(
        case
            when observed_category_status = 'not_evaluable'
                then 1
            else 0
        end
    ) as category_not_evaluable_records,

    sum(
        case
            when observed_category_status = 'not_tested'
                then 1
            else 0
        end
    ) as category_not_tested_records,

    round(
        100.0
        * sum(
            case
                when observed_category_status <> 'not_tested'
                    then 1
                else 0
            end
        )
        / count(*),
        1
    ) as category_testing_coverage_pct

from category_classified

group by
    cohort_id,
    organism_source,
    source_qualifier,
    category_order,
    mdr_category

order by
    cohort_id,
    organism_source,
    category_order

,cohort_id,organism_source,source_qualifier,category_order,mdr_category,culture_organism_records,patients,category_tested_records,category_positive_records,category_observed_non_positive_records,category_not_evaluable_records,category_not_tested_records,category_testing_coverage_pct
0,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,1,extended_spectrum_cephalosporins,9,9,9,1,8,0,0,100.0
1,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,2,fluoroquinolones,9,9,9,4,5,0,0,100.0
2,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,3,aminoglycosides,9,9,9,0,9,0,0,100.0
3,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,4,carbapenems,9,9,9,4,5,0,0,100.0
4,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,5,antipseudomonal_penicillin_combination,9,9,5,1,4,0,4,55.6
5,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,6,siderophore_cephalosporin,9,9,0,0,0,0,9,0.0
6,blood,PSEUDOMONAS AERUGINOSA,None,1,extended_spectrum_cephalosporins,402,377,402,40,361,1,0,100.0
7,blood,PSEUDOMONAS AERUGINOSA,None,2,fluoroquinolones,402,377,402,70,330,2,0,100.0
8,blood,PSEUDOMONAS AERUGINOSA,None,3,aminoglycosides,402,377,363,5,358,0,39,90.3
9,blood,PSEUDOMONAS AERUGINOSA,None,4,carbapenems,402,377,402,47,355,0,0,100.0


The MDR rule counts non-susceptibility across six defined antimicrobial categories. Within a category, an interpretable `Intermediate` or `Resistant` result is positive; untested or conflicting endpoints are kept separate.

### MDR *P. aeruginosa* — assess evaluability

In [7]:
%sql
with category_definitions as (

    select *
    from values

        (
            1,
            'extended_spectrum_cephalosporins',
            'Cefepime'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftazidime'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftazidime/Avibactam'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftolozane/Tazobactam'
        ),

        (
            2,
            'fluoroquinolones',
            'Ciprofloxacin'
        ),
        (
            2,
            'fluoroquinolones',
            'Levofloxacin'
        ),

        (
            3,
            'aminoglycosides',
            'Tobramycin'
        ),

        (
            4,
            'carbapenems',
            'Imipenem'
        ),
        (
            4,
            'carbapenems',
            'Meropenem'
        ),
        (
            4,
            'carbapenems',
            'Imipenem/Relebactam'
        ),

        (
            5,
            'antipseudomonal_penicillin_combination',
            'Piperacillin/Tazobactam'
        ),

        (
            6,
            'siderophore_cephalosporin',
            'Cefiderocol'
        )

    as definitions(
        category_order,
        mdr_category,
        antibiotic
    )

),

categories as (

    select distinct
        category_order,
        mdr_category

    from category_definitions

),

pseudomonas_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Pseudomonas aeruginosa'

),

category_results as (

    select
        p.culture_organism_key,
        p.culture_key,
        p.cohort_id,
        p.patient_key,

        p.organism_source,
        p.source_qualifier,

        d.category_order,
        d.mdr_category,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from pseudomonas_scope p

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on p.culture_organism_key = r.culture_organism_key

    inner join category_definitions d
        on r.antibiotic = d.antibiotic

),

isolate_category_grid as (

    select
        p.culture_organism_key,
        p.culture_key,
        p.cohort_id,
        p.patient_key,

        p.organism_source,
        p.source_qualifier,

        c.category_order,
        c.mdr_category

    from pseudomonas_scope p

    cross join categories c

),

category_summary as (

    select
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,

        g.category_order,
        g.mdr_category,

        count(
            r.antibiotic
        ) as endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as positive_endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Susceptible'
                    then 1
                else 0
            end
        ) as susceptible_endpoint_count,

        sum(
            case
                when r.antibiotic is not null
                 and r.resolution_status in (
                     'missing',
                     'conflict'
                 )
                    then 1

                when r.antibiotic is not null
                 and r.resolution_status = 'resolved'
                 and (
                     r.susceptibility is null
                     or r.susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count

    from isolate_category_grid g

    left join category_results r
        on g.culture_organism_key = r.culture_organism_key
       and g.mdr_category = r.mdr_category

    group by
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,

        g.category_order,
        g.mdr_category

),

category_classified as (

    select
        *,

        case
            when positive_endpoint_count > 0
                then 'positive'

            when endpoint_count = 0
                then 'not_tested'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when susceptible_endpoint_count > 0
                then 'observed_non_positive'

            else 'not_evaluable'
        end as observed_category_status

    from category_summary

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        sum(
            case
                when observed_category_status = 'positive'
                    then 1
                else 0
            end
        ) as positive_category_count,

        sum(
            case
                when observed_category_status =
                     'observed_non_positive'
                    then 1
                else 0
            end
        ) as observed_non_positive_category_count,

        sum(
            case
                when observed_category_status = 'not_evaluable'
                    then 1
                else 0
            end
        ) as not_evaluable_category_count,

        sum(
            case
                when observed_category_status = 'not_tested'
                    then 1
                else 0
            end
        ) as not_tested_category_count,

        array_sort(
            collect_set(
                concat(
                    mdr_category,
                    '=',
                    observed_category_status
                )
            )
        ) as category_statuses

    from category_classified

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier

),

ast_classified as (

    select
        *,

        not_evaluable_category_count
        + not_tested_category_count
            as unknown_category_count,

        positive_category_count
        + not_evaluable_category_count
        + not_tested_category_count
            as maximum_possible_positive_category_count,

        case
            when positive_category_count >= 3
                then 'positive'

            when positive_category_count
               + not_evaluable_category_count
               + not_tested_category_count < 3
                then 'not_detected'

            else 'not_evaluable'
        end as ast_mdr_p_aeruginosa_status

    from isolate_summary

)

select
    cohort_id,
    organism_source,
    source_qualifier,

    ast_mdr_p_aeruginosa_status,

    positive_category_count,
    observed_non_positive_category_count,
    unknown_category_count,
    maximum_possible_positive_category_count,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from ast_classified

group by
    cohort_id,
    organism_source,
    source_qualifier,

    ast_mdr_p_aeruginosa_status,

    positive_category_count,
    observed_non_positive_category_count,
    unknown_category_count,
    maximum_possible_positive_category_count

order by
    cohort_id,
    organism_source,
    ast_mdr_p_aeruginosa_status,
    positive_category_count desc,
    unknown_category_count

,cohort_id,organism_source,source_qualifier,ast_mdr_p_aeruginosa_status,positive_category_count,observed_non_positive_category_count,unknown_category_count,maximum_possible_positive_category_count,culture_organism_records,patients
0,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_detected,1,4,1,2,2,2
1,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_detected,0,5,1,1,1,1
2,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_detected,0,4,2,2,1,1
3,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_evaluable,2,3,1,3,1,1
4,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_evaluable,1,3,2,3,3,3
5,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,positive,3,2,1,4,1,1
6,blood,PSEUDOMONAS AERUGINOSA,None,not_detected,1,4,1,2,55,48
7,blood,PSEUDOMONAS AERUGINOSA,None,not_detected,0,5,1,1,262,252
8,blood,PSEUDOMONAS AERUGINOSA,None,not_detected,0,4,2,2,33,32
9,blood,PSEUDOMONAS AERUGINOSA,None,not_evaluable,2,3,1,3,14,14


At least three positive categories establishes the phenotype. With fewer than three positive categories, I only call the result `not_detected` when the unknown categories could not bring the count up to three. Otherwise it remains `not_evaluable`.

### Audit source MDR terminology

In [8]:
%sql
select
    organism_source,
    source_qualifier,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    array_sort(
        collect_set(cohort_id)
    ) as cohorts

from workspace.micro_dev.int_mapped_cohort_culture_organism

where organism_standardized = 'Pseudomonas aeruginosa'

  and (
         upper(organism_source) like '%MDR%'

      or upper(organism_source)
             like '%MULTI%DRUG%'

      or upper(organism_source)
             like '%MULTIDRUG%'

      or upper(
             coalesce(source_qualifier, '')
         ) like '%MDR%'

      or upper(
             coalesce(source_qualifier, '')
         ) like '%MULTI%DRUG%'

      or upper(
             coalesce(source_qualifier, '')
         ) like '%MULTIDRUG%'
  )

group by
    organism_source,
    source_qualifier

order by
    culture_organism_records desc,
    organism_source

,organism_source,source_qualifier,culture_organism_records,patients,cohorts


This checks whether ARMD already labels organisms as MDR. The source-label audit does not supply an MDR classification for this release, so the model relies on AST-derived categories.

### Audit source ESBL terminology

In [9]:
%sql
select
    organism_source,
    organism_standardized,
    source_qualifier,
    mapping_status,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    array_sort(
        collect_set(cohort_id)
    ) as cohorts

from workspace.micro_dev.int_mapped_cohort_culture_organism

where
       upper(organism_source) like '%ESBL%'

    or upper(organism_source)
           like '%EXTENDED%SPECTRUM%BETA%LACTAMASE%'

    or upper(organism_source)
           like '%EXTENDED%SPECTRUM%B-LACTAMASE%'

    or upper(
           coalesce(source_qualifier, '')
       ) like '%ESBL%'

    or upper(
           coalesce(source_qualifier, '')
       ) like '%EXTENDED%SPECTRUM%BETA%LACTAMASE%'

group by
    organism_source,
    organism_standardized,
    source_qualifier,
    mapping_status

order by
    culture_organism_records desc,
    organism_source

,organism_source,organism_standardized,source_qualifier,mapping_status,culture_organism_records,patients,cohorts


There is no usable source ESBL designation or confirmatory assay here. I therefore use an **extended-spectrum cephalosporin-resistant Enterobacterales (ESCE-R)** category instead of calling it a confirmed ESBL phenotype.

### ESCE-R — defining antibiotic coverage

In [10]:
%sql
with qualifying_antibiotics as (

    select *
    from values

        (1, 'Cefepime'),
        (2, 'Ceftriaxone'),
        (3, 'Cefotaxime'),
        (4, 'Ceftazidime'),
        (5, 'Ceftazidime/Avibactam'),
        (6, 'Ceftolozane/Tazobactam')

    as qualifying(
        antibiotic_order,
        antibiotic
    )

),

organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'ENTEROBACTER AEROGENES%'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA OXYTOCA%'
                then 'Klebsiella oxytoca'

            when organism_source like 'KLEBSIELLA PNEUMONIAE%'
                then 'Klebsiella pneumoniae'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

        end as esce_organism_group

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

eligible_organisms as (

    select *
    from organism_scope

    where esce_organism_group is not null

),

organism_antibiotic_grid as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,

        e.organism_source,
        e.source_qualifier,
        e.esce_organism_group,

        q.antibiotic_order,
        q.antibiotic

    from eligible_organisms e

    cross join qualifying_antibiotics q

),

endpoint_classified as (

    select
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,
        g.esce_organism_group,

        g.antibiotic_order,
        g.antibiotic,

        r.resolution_status,
        r.susceptibility,

        case
            when r.antibiotic is null
                then 'not_tested'

            when r.resolution_status = 'resolved'
             and r.susceptibility = 'Resistant'
                then 'resistant'

            when r.resolution_status = 'resolved'
             and r.susceptibility in (
                 'Susceptible',
                 'Intermediate'
             )
                then 'observed_non_resistant'

            else 'not_evaluable'
        end as observed_endpoint_status

    from organism_antibiotic_grid g

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on g.culture_organism_key = r.culture_organism_key
       and g.antibiotic = r.antibiotic

)

select
    cohort_id,
    esce_organism_group,
    organism_source,
    source_qualifier,

    antibiotic_order,
    antibiotic,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    sum(
        case
            when observed_endpoint_status <> 'not_tested'
                then 1
            else 0
        end
    ) as tested_endpoint_records,

    sum(
        case
            when observed_endpoint_status in (
                'resistant',
                'observed_non_resistant'
            )
                then 1
            else 0
        end
    ) as evaluable_endpoint_records,

    sum(
        case
            when observed_endpoint_status = 'resistant'
                then 1
            else 0
        end
    ) as resistant_records,

    sum(
        case
            when observed_endpoint_status =
                 'observed_non_resistant'
                then 1
            else 0
        end
    ) as observed_non_resistant_records,

    sum(
        case
            when observed_endpoint_status = 'not_evaluable'
                then 1
            else 0
        end
    ) as not_evaluable_records,

    sum(
        case
            when observed_endpoint_status = 'not_tested'
                then 1
            else 0
        end
    ) as not_tested_records,

    round(
        100.0
        * sum(
            case
                when observed_endpoint_status <> 'not_tested'
                    then 1
                else 0
            end
        )
        / count(*),
        1
    ) as testing_coverage_pct,

    round(
        100.0
        * sum(
            case
                when observed_endpoint_status in (
                    'resistant',
                    'observed_non_resistant'
                )
                    then 1
                else 0
            end
        )
        / count(*),
        1
    ) as evaluable_coverage_pct

from endpoint_classified

group by
    cohort_id,
    esce_organism_group,
    organism_source,
    source_qualifier,

    antibiotic_order,
    antibiotic

order by
    cohort_id,
    esce_organism_group,
    organism_source,
    antibiotic_order

,cohort_id,esce_organism_group,organism_source,source_qualifier,antibiotic_order,antibiotic,culture_organism_records,patients,tested_endpoint_records,evaluable_endpoint_records,resistant_records,observed_non_resistant_records,not_evaluable_records,not_tested_records,testing_coverage_pct,evaluable_coverage_pct
0,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,1,Cefepime,1,1,1,1,0,1,0,0,100.0,100.0
1,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,2,Ceftriaxone,1,1,1,1,0,1,0,0,100.0,100.0
2,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,3,Cefotaxime,1,1,1,1,0,1,0,0,100.0,100.0
3,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,4,Ceftazidime,1,1,0,0,0,0,0,1,0.0,0.0
4,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,5,Ceftazidime/Avibactam,1,1,0,0,0,0,0,1,0.0,0.0
5,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,6,Ceftolozane/Tazobactam,1,1,0,0,0,0,0,1,0.0,0.0
6,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,1,Cefepime,1,1,1,1,0,1,0,0,100.0,100.0
7,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,2,Ceftriaxone,1,1,1,1,0,1,0,0,100.0,100.0
8,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,3,Cefotaxime,1,1,0,0,0,0,0,1,0.0,0.0
9,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,4,Ceftazidime,1,1,1,1,0,1,0,0,100.0,100.0


The defining drugs are cefepime, ceftriaxone, cefotaxime, ceftazidime, ceftazidime/avibactam, and ceftolozane/tazobactam. The eligible groups are reviewed *E. coli*, *K. aerogenes*, *K. oxytoca*, *K. pneumoniae*, and *Enterobacter*.

### ESCE-R — derive the status

In [11]:
%sql
with organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'ENTEROBACTER AEROGENES%'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA OXYTOCA%'
                then 'Klebsiella oxytoca'

            when organism_source like 'KLEBSIELLA PNEUMONIAE%'
                then 'Klebsiella pneumoniae'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

        end as esce_organism_group

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

eligible_organisms as (

    select *
    from organism_scope

    where esce_organism_group is not null

),

qualifying_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,

        e.organism_source,
        e.source_qualifier,
        e.esce_organism_group,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from eligible_organisms e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key

       and r.antibiotic in (
           'Cefepime',
           'Ceftriaxone',
           'Cefotaxime',
           'Ceftazidime',
           'Ceftazidime/Avibactam',
           'Ceftolozane/Tazobactam'
       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,
        esce_organism_group,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as qualifying_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when antibiotic is not null
                 and resolution_status in (
                     'missing',
                     'conflict'
                 )
                    then 1

                when antibiotic is not null
                 and resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as qualifying_esce_results

    from qualifying_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,
        esce_organism_group

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_esce_enterobacterales_status

    from isolate_summary

)

select
    cohort_id,
    esce_organism_group,
    organism_source,
    source_qualifier,

    ast_esce_enterobacterales_status,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    round(
        100.0
        * count(*)
        / sum(count(*)) over (
            partition by
                cohort_id,
                esce_organism_group
        ),
        1
    ) as pct_within_cohort_organism_group

from ast_classified

group by
    cohort_id,
    esce_organism_group,
    organism_source,
    source_qualifier,
    ast_esce_enterobacterales_status

order by
    cohort_id,
    esce_organism_group,
    organism_source,
    ast_esce_enterobacterales_status

,cohort_id,esce_organism_group,organism_source,source_qualifier,ast_esce_enterobacterales_status,culture_organism_records,patients,pct_within_cohort_organism_group
0,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,not_detected,1,1,0.3
1,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,not_detected,1,1,0.3
2,blood,Enterobacter spp.,ENTEROBACTER CLOACAE,None,not_detected,55,54,16.6
3,blood,Enterobacter spp.,ENTEROBACTER CLOACAE,None,positive,8,8,2.4
4,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,None,not_detected,200,189,60.4
5,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,None,not_evaluable,17,17,5.1
6,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,None,positive,37,36,11.2
7,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX (CARBAPENEM RESISTANT),carbapenem_resistant,positive,3,3,0.9
8,blood,Enterobacter spp.,ENTEROBACTER GERGOVIAE,None,not_detected,4,4,1.2
9,blood,Enterobacter spp.,ENTEROBACTER SPECIES,None,not_detected,5,5,1.5


### Decision

The phenotype definitions use reconciled categorical AST rather than source wording. CRE and ESCE-R are derived independently; the reporting category gives CRE precedence. CRE/MDR reporting records remain in the upstream evidence and the separate MDRO mart, while qualifying ordinary and resistance categories feed the main susceptibility mart.

These exploratory queries preserve the rule-development record. Production logic lives in the dbt phenotype and reporting models; Notebook 09 inspects their output, and Notebook 13 documents the independently selected reporting populations.